# K-Means 聚类问题 (MSSC)

**类别：** 选址

来源：[https://www.hexaly.com/templates/k-means-clustering-problem-mssc](https://www.hexaly.com/templates/k-means-clustering-problem-mssc)


## 问题

**在 K-Means 聚类问题**（最小平方和聚类，MSSC）中，我们希望将一组多维观测点划分为 k 个聚类，每个聚类由其重心定义。每个观测点属于具有最近重心的聚类。更多细节，请参阅 [Wikipedia](http://en.wikipedia.org/wiki/K-means_clustering)。

	

### 学到的建模原则

- 添加 [set decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模聚类
- 使用 [lambda functions](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算每个聚类的重心和方差


## 数据

数据格式如下：

- 第 1 行：观测点数和每个观测点的维度数
- 对每个观测点：每个维度上的坐标以及其在最优解中所属的聚类


## 模型

K-Means 聚类问题 (MSSC) 的Hexaly模型使用 [set decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)。每个 set variable 表示一个聚类，其内部的元素表示属于该聚类的观测点。我们使用一个 [**partition** operator](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html#n-ary-operators) 来确保每个观测点恰好属于一个聚类。

我们使用 [lambda function](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 对每个聚类内所有观测点在所有维度上的坐标应用 **sum** 算子来计算每个聚类的重心。注意，此求和中项的数量在搜索过程中会随着集合的大小变化而变化。

然后我们可以计算总方差。一个聚类的方差是该聚类重心与每个观测点之间的欧几里得距离平方之和。与重心类似，我们使用 lambda function 计算每个聚类的方差。目标是最小化这些方差之和。


## Results

**Hexaly Optimizer 在 1 分钟运行时间内即可在 K-Means 聚类问题 (MSSC) 上达到低于 1% 的 gap**，这些实例来自 UCI 机器学习仓库和 TSPLIB 研究基准，包含超过 **10,000 个观测点**。我们的 [K-Means Clustering Problem (MSSC) benchmark page](https://www.hexaly.com/benchmark/hexaly-vs-gurobi-k-means-clustering-mssc) 展示了 Hexaly Optimizer 在这一富有挑战性的问题上如何超越 Gurobi 等传统的通用优化求解器。

[Explore this benchmark](https://www.hexaly.com/benchmark/hexaly-vs-gurobi-k-means-clustering-mssc)


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

def read_elem(filename):
    with open(filename) as f:
        return [str(elem) for elem in f.read().split()]

#
# Read instance data
#
def read_instance(filename):
    file_it = iter(read_elem(filename))

    # Data properties
    nb_observations = int(next(file_it))
    nb_dimensions = int(next(file_it))

    coordinates_data = [None] * nb_observations
    for o in range(nb_observations):
        coordinates_data[o] = [None] * (nb_dimensions)
        for d in range(nb_dimensions):
            coordinates_data[o][d] = float(next(file_it))
        next(file_it) # skip initial clusters

    return nb_observations, nb_dimensions, coordinates_data

def main(instance_file, output_file, time_limit, k):
    nb_observations, nb_dimensions, coordinates_data = read_instance(instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # clusters[c] represents the points in cluster c
        clusters = [model.set(nb_observations) for c in range(k)]

        # Each point must be in one cluster and one cluster only
        model.constraint(model.partition(clusters))

        # Coordinates of points
        coordinates = model.array(coordinates_data)

        # Compute variances
        variances = []
        for cluster in clusters:
            size = model.count(cluster)

            # Compute centroid of cluster
            centroid = [0 for d in range(nb_dimensions)]
            for d in range(nb_dimensions):
                coordinate_lambda = model.lambda_function(
                    lambda i: model.at(coordinates, i, d))
                centroid[d] = model.iif(
                    size == 0,
                    0,
                    model.sum(cluster, coordinate_lambda) / size)

            # Compute variance of cluster
            variance = model.sum()
            for d in range(nb_dimensions):
                dimension_variance_lambda = model.lambda_function(lambda i:
                    model.pow(model.at(coordinates, i, d) - centroid[d], 2))
                dimension_variance = model.sum(cluster, dimension_variance_lambda)
                variance.add_operand(dimension_variance)
            variances.append(variance)

        # Minimize the total variance
        obj = model.sum(variances)
        model.minimize(obj)

        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = time_limit

        optimizer.solve()

        #
        # Write the solution in a file in the following format:
        #  - objective value
        #  - k
        #  - for each cluster, a line with the elements in the cluster
        #    (separated by spaces)
        #
        if output_file != None:
            with open(output_file, 'w') as f:
                f.write("%f\n" % obj.value)
                f.write("%d\n" % k)
                for c in range(k):
                    for o in clusters[c].value:
                        f.write("%d " % o)
                    f.write("\n")

if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python kmeans.py inputFile [outputFile] [timeLimit] [k value]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 60
    k = int(sys.argv[4]) if len(sys.argv) >= 5 else 2
    main(instance_file, output_file, time_limit, k)
